# LOSO completo en Colab T4 — STFT y CWT

Notebook autocontenido para correr el LOSO **full** (epochs=50, batch=128, AMP) en una GPU T4 de Colab.

**Pre-requisito**: subir el directorio `data/derivatives/modspec_stft_200/` y `data/derivatives/modspec_cwt_200/` a tu Google Drive (~11 GB total). El preproceso y modspec ya están hechos localmente.

Estimado: **~20 min STFT + 20 min CWT = 40 min total** en T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar deps

In [ ]:
!git clone https://github.com/spalaciobe/tps-alzheimer-modspec.git
%cd tps-alzheimer-modspec
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Montar Google Drive y enlazar `data/derivatives/`

Ajusta `DRIVE_DERIV_PATH` a la ruta donde subiste los HDF5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DERIV_PATH = '/content/drive/MyDrive/tps-alzheimer/derivatives'  # ← AJUSTAR

import os
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/derivatives'):
    os.symlink(DRIVE_DERIV_PATH, 'data/derivatives')

!ls -la data/derivatives/

## 4. (Opcional) Copiar HDF5 al SSD efímero — más rápido

Si Drive es lento, copia los `.h5` a `/content/data/derivatives/` (~2 min, da ~5× boost en data loading).

In [ ]:
# Descomenta si quieres copiar a SSD local
# !mkdir -p data_local/derivatives
# !cp -r data/derivatives/modspec_stft_200 data_local/derivatives/
# !cp -r data/derivatives/modspec_cwt_200 data_local/derivatives/
# !rm data/derivatives && ln -s /content/tps-alzheimer-modspec/data_local/derivatives data/derivatives

## 5. LOSO STFT (full) — ~20 min en T4

In [ ]:
!python scripts/03_train_loso.py --method stft --fs 200 --seed 0

## 6. LOSO CWT (full)

In [ ]:
!python scripts/03_train_loso.py --method cwt --fs 200 --seed 0

## 7. Saliency + SVM + Comparación + Figuras

In [ ]:
!python scripts/04_extract_saliency_features.py --method stft --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20
!python scripts/04_extract_saliency_features.py --method cwt  --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20
!python scripts/04_extract_saliency_features.py --method stft --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20 --saliency-method vanilla
!python scripts/04_extract_saliency_features.py --method cwt  --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20 --saliency-method vanilla

In [ ]:
!python scripts/05_run_svm.py --method stft --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80
!python scripts/05_run_svm.py --method cwt  --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80
!python scripts/05_run_svm.py --method stft --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80 --saliency-method vanilla
!python scripts/05_run_svm.py --method cwt  --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80 --saliency-method vanilla

In [ ]:
!python scripts/06_compare_stft_cwt.py --classifier cnn
!python scripts/06_compare_stft_cwt.py --classifier svm --per-fold
!python scripts/06_compare_stft_cwt.py --classifier svm --per-fold --saliency-method vanilla
!python scripts/07_generate_figures.py --per-fold
!python scripts/07_generate_figures.py --per-fold --saliency-method vanilla

## 8. Copiar resultados de vuelta a Drive

In [ ]:
import shutil
DRIVE_RESULTS = '/content/drive/MyDrive/tps-alzheimer/results-full'
shutil.copytree('results', DRIVE_RESULTS, dirs_exist_ok=True)
shutil.copytree('data/derivatives/saliency', DRIVE_RESULTS + '/saliency', dirs_exist_ok=True)
print('Resultados full guardados en', DRIVE_RESULTS)